# 🚔 Incident-Triggered Multi-Camera Vehicle Search (Live GPU Demonstration)

This notebook provides the complete, GPU-accelerated interactive web application for:
1. **Real-time video tracking**: Live YOLO11 + ByteTrack vehicle identification badges on camera streams.
2. **Feature extraction**: FastReID SBS(R50-ibn) representation.
3. **Incident retrieval**: Cross-camera search fusing appearance cosine similarity and temporal/graph priors.

---

### Step 1: Clone Repository & Set Up GPU Runtime

In [ ]:
# Clone or pull latest main
!if [ -d "Vehicle-Tracking" ]; then cd Vehicle-Tracking && git pull origin main; else git clone https://github.com/neoaltair/Vehicle-Tracking.git; fi
%cd /content/Vehicle-Tracking

# Check CUDA GPU availability
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

### Step 2: Install Dependencies & FastReID (Pure-Python Mode)

In [ ]:
# Install project in editable mode with development tools
!pip install -q -e ".[dev]"

# Install pure-python FastReID release without conflicting PyTorch downgrades
!pip install -q fastreid==1.4.0 --no-deps

# Install localtunnel to expose the Streamlit UI via a public web URL
!npm install -g -q localtunnel

### Step 3: Download Pretrained FastReID Re-Identification Weights

In [ ]:
import os
from pathlib import Path

weights_dir = Path("outputs/weights")
weights_dir.mkdir(parents=True, exist_ok=True)
weights_file = weights_dir / "veri_sbs_R50-ibn.pth"

if not weights_file.exists():
    print("Downloading VeRi-776 SBS(R50-ibn) checkpoint...")
    !wget -q -O outputs/weights/veri_sbs_R50-ibn.pth https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth
    print("Downloaded successfully!")
else:
    print("Weights already exist at", weights_file)

### Step 4: Verify Sample Traffic Videos

In [ ]:
!ls -lh data/raw/samples/

### Step 5: Launch Live Interactive Surveillance Web App

Run the cell below:
1. Note the **External IP** displayed below (you will paste this into the tunnel password prompt).
2. Click the `localtunnel.me` link generated.
3. Paste the IP into "Tunnel Password" and submit.

In [ ]:
import urllib.request

# Display Colab instance external IP for localtunnel authorization
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"🔑 Your LocalTunnel Password / Endpoint IP is: {external_ip}\n")

# Start Streamlit app in background and expose via localtunnel
!streamlit run app.py --server.port 8501 --server.headless true & npx localtunnel --port 8501